# Chapter 1: Chunking

Your RAG system returns the wrong answer and you blame the model. But the real problem happened before any search ran. You chunked your documents wrong.

In this notebook you will split the same refund policy two different ways and watch the same question produce two very different results. The only thing that changes is one number. Chunk size.

Run every cell top to bottom. When you reach the "Try it yourself" section, change the numbers and rerun.

## One time setup, getting your API keys

This is the only chapter that walks through API keys. Every other chapter assumes the keys are already in Colab secrets and picks them up automatically.

You need three keys. All three have free tiers that cover the entire series. Total cost across all seven chapters is under two dollars.

1. **OpenAI** at [platform.openai.com](https://platform.openai.com). New accounts get five dollars of free credit. Copy your key.
2. **LangSmith** at [smith.langchain.com](https://smith.langchain.com). Sign up, go to Settings, API Keys, create a personal key. Free tier gives you five thousand traces per month.
3. **Cohere** at [dashboard.cohere.com](https://dashboard.cohere.com). Sign up, go to API Keys, copy your trial key. Free tier gives you one thousand rerank calls per month.

Now add them to Colab secrets. Click the key icon on the left sidebar of Colab. Add three secrets:

- `OPENAI_API_KEY` set to your OpenAI key
- `LANGCHAIN_API_KEY` set to your LangSmith key
- `COHERE_API_KEY` set to your Cohere key

Toggle notebook access on for all three. You only do this once. Every chapter from this point on picks the keys up automatically.

In [ ]:
!pip install -q \
    langchain==0.3.14 \
    langchain-openai==0.2.14 \
    langchain-community==0.3.14 \
    langchain-chroma==0.1.4 \
    langchain-text-splitters==0.3.5 \
    chromadb==0.5.23 \
    pypdf==5.1.0 \
    langsmith==0.2.6 \
    pandas==2.2.3

In [ ]:
import os, sys
if not os.path.exists("rag-for-pms"):
    !git clone -q https://github.com/DDRXV/rag-for-pms.git
%cd rag-for-pms
sys.path.insert(0, os.getcwd())

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your three API keys out of Colab secrets and turned on LangSmith tracing. From now on every embedding call, every chat call, and every retrieval step gets logged automatically to a clickable waterfall trace.

At the end of this notebook you can open [smith.langchain.com](https://smith.langchain.com) and see your full pipeline as a visual timeline. No code reading required.

## Look at the corpus

You are about to load six SkillAgents AI documents. The one that matters for this chapter is `refund_policy.pdf`. Your running question for the whole notebook is:

> How much of my subscription fee is refunded if I cancel my Pro Annual plan after five months of use?

The correct answer lives in Section 4.2 of the refund policy, where the pro-rated calculation is spelled out with the two month deduction and the fifty dollar processing fee. Your job for the rest of this notebook is to chunk the corpus in a way that lets vector search actually find that section.

In [ ]:
from utils.shared import load_corpus

docs = load_corpus()

for d in docs[:6]:
    print(f"  {d.metadata.get('source', 'unknown'):25s}  {len(d.page_content):5d} chars")

## The naive approach, one big chunk

First we try the lazy thing. Chunk every document into blocks of 5000 characters with no overlap. The refund policy is only 2617 characters long, so the entire document becomes one single chunk. That chunk contains the overview, the free trial section, the student plan, the pro annual calculation, the team plan, the enterprise note, and the how-to-request-a-refund section. Everything in one undifferentiated blob.

Then we search with our question and see what comes back.

In [ ]:
from utils.shared import build_index, search, show_results

# One huge chunk per document. This is what most people do by accident.
chunk_size = 5000

naive_index = build_index(chunk_size=chunk_size, chunk_overlap=0)
question = "How much of my subscription fee is refunded if I cancel my Pro Annual plan after five months of use?"

naive_results = search(naive_index, question, k=3)
show_results(naive_results, question=question)

## What went wrong

Look at the top row. The source is `refund_policy.pdf`. Good, the retriever found the right document. But now look at the score column. The score is around 0.9 or higher under L2 distance, which is mediocre. The embedding for a chunk that covers five different refund categories is a muddled signal. It is close-ish to the question about Pro Annual refunds, but it is also close-ish to questions about Free Trial, Student, Team, and Enterprise refunds.

Now look at the preview. The chunk starts with "SkillAgents AI Refund Policy. Last updated..." which is the title of the document, not the pro-rated calculation. The LLM receiving this chunk would have to read through the whole policy to find the Pro Annual section and then do the math. That is exactly the kind of noisy context that causes hallucinations.

## The fix, smaller chunks

Now we rebuild the index with `chunk_size=300` and `chunk_overlap=50`. The same refund policy becomes many smaller pieces. Each piece holds roughly one paragraph of focused content.

When we search the same question, the match should point directly at the Section 4.2 pro-rated calculation.

In [ ]:
# Smaller chunks means each one is tightly scoped. Overlap keeps context across boundaries.
chunk_size = 300
chunk_overlap = 50

better_index = build_index(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
better_results = search(better_index, question, k=3)
show_results(better_results, question=question)

## What got better

The top result is now a focused chunk from Section 4.2. The score drops substantially, somewhere in the 0.6 to 0.7 range. The preview column shows the exact pro-rated calculation text. Look at the top three results together. All three are now from `refund_policy.pdf` and all three are adjacent paragraphs covering cancellation and refund mechanics.

This is the entire chunking lesson.

- Same document.
- Same question.
- Same embedding model.
- Different chunk size.
- Dramatically different retrieval quality.

An LLM given the top chunk can answer the question in one sentence with the exact numbers. That is the difference between a RAG demo and a production RAG system.

## Open your LangSmith trace

Go to [smith.langchain.com](https://smith.langchain.com). Click into the project called `rag-for-pms`. Open the most recent trace.

You will see two `similarity_search_with_score` calls side by side. The first was the `chunk_size=5000` run, returning one diluted mega-chunk per document. The second was the `chunk_size=300` run, returning focused paragraph-sized chunks. Click into each one and you can see the exact chunks, the exact distance scores, the embedding call cost, and the latency.

This is the pipeline made visible. You did not have to read a single line of code to understand what happened. That is the point of LangSmith, and it is why we use it in every chapter.

## Try it yourself

Pick one of these experiments, change the numbers in the cells above, and rerun. See what happens.

1. Change `chunk_size` in the better_index cell to **100**. Rerun. Notice the chunks are now so small that each one is missing the context it needs to answer the question. The top score gets worse because no single fragment holds the full calculation.
2. Change `chunk_size` in the better_index cell to **1000**. Rerun. See if the score improves further or stays about the same. Somewhere between 300 and 1000 is the sweet spot for this corpus.
3. Change the `question` variable to `"What is the price of the Team plan?"`. Rerun the better_index search cell. Watch the top match switch from `refund_policy.pdf` to `pricing.pdf` without changing any other code.

After each experiment, open LangSmith and compare the traces. The visual difference between a good chunk and a bad chunk jumps out fast when you can see them side by side.

## Homework

Three longer exercises you can do on your own. Each one takes ten to twenty minutes.

1. **Try a different splitter.** Replace the default splitter with `MarkdownHeaderTextSplitter` from `langchain_text_splitters`. Split `product_guide.md` by its H2 headers. Compare the chunks you get against the default `RecursiveCharacterTextSplitter`. Which approach gives cleaner boundaries for a markdown document?

2. **Measure chunk quality programmatically.** For chunk sizes 100, 300, 500, 1000, and 2000, run the same five questions against each index and record the top-1 score. Plot the results. Where is the sweet spot for this corpus? Is it the same sweet spot for every question?

3. **Apply it to your own company.** Pick a real document from your company (a refund policy, a product spec, an onboarding guide). Run it through the same pipeline. What chunk size would you start with and why? Write a one paragraph answer with concrete numbers and share it in the cohort Slack.

## What is next

In Chapter 2 we fix a different problem. The user asks a vague question and vector search returns the wrong document even after perfect chunking. The fix is called query translation. You rewrite the question into three more specific versions before you search. It sounds like a small change. The retrieval quality jumps from useless to production ready.

See you there.